# EDA — Amazon Reviews Multi EN

Exploratory Data Analysis untuk dataset `SetFit/amazon_reviews_multi_en`.
Tujuan: pahami distribusi label, panjang teks, dan contoh sampel sebelum modeling.

In [ ]:
from datasets import load_from_disk
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ds = load_from_disk('../data/amazon_reviews_multi_en')
train, val, test = ds['train'], ds['validation'], ds['test']
print(f'train={len(train)}  val={len(val)}  test={len(test)}')

## 1. Distribusi Label (5 bintang)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, split) in zip(axes, [('Train', train), ('Validation', val), ('Test', test)]):
    c = Counter(split['label'])
    ax.bar([str(k) for k in sorted(c)], [c[k] for k in sorted(c)], color='steelblue')
    ax.set_title(f'{name} (n={len(split)})')
    ax.set_xlabel('Star (0-4)'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()
print('Semua split seimbang: 20% per bintang')

## 2. Distribusi Setelah Mapping (3 kelas)

In [ ]:
def to_sentiment(label):
    return 0 if label in (0,1) else (1 if label==2 else 2)

mapped = Counter(to_sentiment(l) for l in train['label'])
labels = ['NEGATIVE', 'NEUTRAL', 'POSITIVE']
counts = [mapped[0], mapped[1], mapped[2]]

plt.figure(figsize=(7,4))
bars = plt.bar(labels, counts, color=['#dc3545','#ffc107','#28a745'])
for b, c in zip(bars, counts):
    plt.text(b.get_x()+b.get_width()/2, b.get_height(), f'{c:,}', ha='center', va='bottom')
plt.title('Train set setelah mapping — rasio 2:1:2 (NEUTRAL minoritas)')
plt.ylabel('Count'); plt.show()
print(f'NEG={mapped[0]:,}  NEU={mapped[1]:,}  POS={mapped[2]:,}')

## 3. Panjang Teks

In [ ]:
lens = np.array([len(t.split()) for t in train['text'][:10000]])
plt.figure(figsize=(9,4))
plt.hist(lens, bins=60, color='steelblue', edgecolor='white')
plt.axvline(128, color='red', linestyle='--', label='max_len=128')
plt.title('Distribusi panjang review (kata)'); plt.xlabel('Words'); plt.legend(); plt.show()
print(f'median={np.median(lens):.0f}  p95={np.percentile(lens,95):.0f}  max={lens.max()}')
print(f'{(lens<=128).mean()*100:.1f}% review <= 128 kata')

## 4. Contoh Sampel per Kelas

In [ ]:
names = {0:'NEGATIVE',1:'NEUTRAL',2:'POSITIVE'}
seen = {0:0,1:0,2:0}
for r in train:
    s = to_sentiment(r['label'])
    if seen[s] < 2:
        print(f'[{names[s]} | bintang {r["label"]}] {r["text"][:120]}...\n')
        seen[s]+=1
    if all(v>=2 for v in seen.values()): break

## Kesimpulan EDA

- Dataset seimbang per bintang (40k/bintang), tapi setelah mapping jadi **imbalance 2:1:2** (NEUTRAL minoritas) — pakai `class_weight='balanced'` dan laporkan F1 per kelas.
- **98.3% review <= 128 kata** -> `max_len=128` tepat, hanya 1.7% terpotong.
- NEUTRAL diperkirakan jadi kelas terlemah (terbukti di baseline: F1=0.46).